# Project 64 — Local Tool Selection Benchmark

**Evaluate whether an agent picks the right tool for each task — locally.**

| Component | Choice |
|-----------|--------|
| LLM | Ollama (qwen3:8b) |
| Framework | LangChain |
| Interface | Jupyter |

## 1 · What You Will Learn

1. Build a **tool selection test harness** with ground-truth labels
2. Measure **tool selection accuracy** across question types
3. Analyze **confusion patterns** (which tools get mixed up?)
4. Compare **prompt strategies** for improving tool selection

## 2 · Why This Project Matters

Agents fail silently when they pick the wrong tool. A benchmark lets you
systematically test routing accuracy before deployment.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## 4 · Imports

In [ ]:
import json
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Tool Catalog and Test Cases

In [ ]:
TOOLS = {'calculator': 'Math calculations', 'web_search': 'Search internet',
         'calendar': 'Schedule management', 'email': 'Send/read emails',
         'file_search': 'Find files', 'code_runner': 'Execute Python',
         'none': 'Answer from general knowledge'}

TEST_CASES = [
    {'query': 'What is 15% of $2,340?', 'expected': 'calculator'},
    {'query': 'What happened in the news today?', 'expected': 'web_search'},
    {'query': 'Do I have meetings tomorrow?', 'expected': 'calendar'},
    {'query': 'Send follow-up email to Sarah', 'expected': 'email'},
    {'query': 'Find the quarterly report PDF', 'expected': 'file_search'},
    {'query': 'Run this Python script', 'expected': 'code_runner'},
    {'query': 'What is the capital of France?', 'expected': 'none'},
    {'query': 'Convert 72F to Celsius', 'expected': 'calculator'},
    {'query': 'Who won the latest World Cup?', 'expected': 'web_search'},
    {'query': 'Schedule a call on Friday', 'expected': 'calendar'},
    {'query': 'Reply to the invoice email', 'expected': 'email'},
    {'query': 'Explain recursion in programming', 'expected': 'none'},
    {'query': 'What is 2^10 + 3^5?', 'expected': 'calculator'},
    {'query': 'Find all .csv files in data/', 'expected': 'file_search'},
]
print(f'{len(TOOLS)} tools, {len(TEST_CASES)} test cases')

## 6 · Tool Selection Agent

In [ ]:
tool_desc = '\n'.join([f'- {n}: {d}' for n,d in TOOLS.items()])

def select_tool(query):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Select the best tool. Available:\n{tools}\nReturn ONLY JSON: {"tool": "name", "confidence": 1-5}'),
        ('human', '{query}'),
    ])
    raw = (prompt | llm | StrOutputParser()).invoke({'tools': tool_desc, 'query': query})
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        if s >= 0: return json.loads(raw[s:e])
    except: pass
    return {'tool': 'none', 'confidence': 1}

print(f'Test: {select_tool("What is 2+2?")}')

## 7 · Run Benchmark

In [ ]:
results = []
for tc in TEST_CASES:
    pred = select_tool(tc['query'])
    predicted = pred.get('tool', 'none').lower().strip()
    correct = predicted == tc['expected']
    results.append({'query': tc['query'][:50], 'expected': tc['expected'],
                    'predicted': predicted, 'correct': correct})
    print(f'  [{"PASS" if correct else "FAIL"}] {tc["query"][:40]:40s} exp={tc["expected"]:12s} got={predicted}')
print(f'Total: {len(results)} tests')

## 8 · Analyze Results

In [ ]:
df = pd.DataFrame(results)
accuracy = df['correct'].mean()
print(f'OVERALL ACCURACY: {accuracy:.1%}')
print('\nACCURACY BY TOOL:')
for tool in sorted(df['expected'].unique()):
    sub = df[df['expected'] == tool]
    print(f'  {tool:15s}: {sub["correct"].mean():.0%} ({sub["correct"].sum()}/{len(sub)})')

print('\nCONFUSION MATRIX:')
print(pd.crosstab(df['expected'], df['predicted'], margins=True).to_string())

## 9 · Save Results

In [ ]:
df.to_csv('tool_selection_results.csv', index=False)
print('Saved.')

## 10 · Limitations & Improvements

- Simulated tools (no execution)
- No multi-tool queries
- Add ambiguous edge cases
- Compare few-shot vs zero-shot routing

## 11 · Key Takeaways

- Tool selection accuracy is measurable with ground-truth benchmarks
- Confusion matrices reveal which tools get mixed up
- Prompt engineering directly impacts routing reliability
- Local benchmarking enables fast iteration